<a href="https://colab.research.google.com/github/Mithilesh-Kr-Chaudhary/machine-learning-projects/blob/main/sms_spam_classifier_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS Spam Classifier

Upload an SMS dataset, train three TF-IDF models (Naive Bayes, Logistic Regression, and Linear SVM), compare them, and classify new messages.

**Expected data:** a CSV/TSV file with one label column (e.g. `label`, `v1`, `class`) and one message column (e.g. `message`, `v2`, `text`). The common `spam.csv` and UCI `SMSSpamCollection` formats both work.

In [ ]:
import io
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import joblib

RANDOM_STATE = 42

In [ ]:
# Upload your dataset file. Click the upload button after running this cell.
uploaded = files.upload()
filename = next(iter(uploaded))
print(f'Uploaded: {filename}')

In [ ]:
# Load common SMS-spam dataset layouts and standardize to: label, message
raw = uploaded[filename]
try:
    data = pd.read_csv(io.BytesIO(raw), sep=None, engine='python', encoding='utf-8')
except UnicodeDecodeError:
    data = pd.read_csv(io.BytesIO(raw), sep=None, engine='python', encoding='latin-1')

# UCI's SMSSpamCollection commonly has no header; reload it when needed.
if len(data.columns) < 2 or not any(str(c).lower() in ['label', 'v1', 'class', 'category', 'type'] for c in data.columns):
    try:
        data = pd.read_csv(io.BytesIO(raw), sep='\t', header=None, names=['label', 'message'], encoding='utf-8')
    except UnicodeDecodeError:
        data = pd.read_csv(io.BytesIO(raw), sep='\t', header=None, names=['label', 'message'], encoding='latin-1')

normalized = {str(c).strip().lower(): c for c in data.columns}
label_candidates = ['label', 'v1', 'class', 'category', 'type']
message_candidates = ['message', 'v2', 'text', 'sms', 'body', 'content']
label_col = next((normalized[c] for c in label_candidates if c in normalized), data.columns[0])
message_col = next((normalized[c] for c in message_candidates if c in normalized), data.columns[1])

df = data[[label_col, message_col]].copy()
df.columns = ['label', 'message']
df = df.dropna().drop_duplicates()
df['label'] = df['label'].astype(str).str.strip().str.lower()
df['message'] = df['message'].astype(str)

# Accept usual names and convert labels to 0 = legitimate, 1 = spam.
spam_names = {'spam', '1', 'yes', 'true', 'junk'}
df['target'] = df['label'].isin(spam_names).astype(int)
if df['target'].nunique() != 2:
    raise ValueError(f'Could not find both spam and legitimate labels. Labels found: {sorted(df.label.unique())}')

print(f'Dataset size: {len(df):,} messages')
display(df.head())
display(df['label'].value_counts())

In [ ]:
# Basic text cleanup. TF-IDF learns useful word and phrase patterns from this text.
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'\d+', ' NUMBER ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_message'] = df['message'].map(clean_text)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_message'], df['target'], test_size=0.20,
    stratify=df['target'], random_state=RANDOM_STATE
)
print(f'Training: {len(X_train):,} | Testing: {len(X_test):,}')

In [ ]:
# Train several classifiers with the same TF-IDF features for a fair comparison.
def make_pipeline(model):
    return Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ('model', model),
    ])

models = {
    'Naive Bayes': make_pipeline(MultinomialNB(alpha=0.5)),
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    'Linear SVM': make_pipeline(LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
}

results, trained_models = [], {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average='binary', zero_division=0)
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, predictions), 'Spam Precision': precision, 'Spam Recall': recall, 'Spam F1': f1})
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('Spam F1', ascending=False).reset_index(drop=True)
display(results_df.style.format({'Accuracy': '{:.2%}', 'Spam Precision': '{:.2%}', 'Spam Recall': '{:.2%}', 'Spam F1': '{:.2%}'}))
best_name = results_df.loc[0, 'Model']
best_model = trained_models[best_name]
print(f'Best model by spam F1: {best_name}')

In [ ]:
# Evaluate the selected model. Spam is class 1; legitimate is class 0.
best_predictions = best_model.predict(X_test)
print(classification_report(y_test, best_predictions, target_names=['legitimate', 'spam'], digits=3))

cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['legitimate', 'spam'], yticklabels=['legitimate', 'spam'])
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

In [ ]:
# Try the trained classifier on your own SMS text.
new_messages = [
    'Congratulations! You have won a free prize. Call now to claim it.',
    'Hi, are we still meeting at 6 pm today?',
]

new_predictions = best_model.predict([clean_text(msg) for msg in new_messages])
for message, prediction in zip(new_messages, new_predictions):
    print(f'[{"SPAM" if prediction == 1 else "LEGITIMATE"}] {message}')

In [ ]:
# Optional: download the selected model for reuse outside this notebook.
joblib.dump({'model': best_model, 'model_name': best_name}, 'sms_spam_classifier.joblib')
files.download('sms_spam_classifier.joblib')